In [1]:
import pandas as pd
import random
import os

In [2]:
# -------------------- Parameters --------------------
INPUT_FILE = "../../solutions/bks/bks_table.csv"   # or .txt
OUTPUT_PREFIX = "stratified_sample"
N_PER_CLASS = 5                # number of instances per stratum
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

In [ ]:
# -------------------- Read file --------------------
def read_instances(file_path):
    """
    Read the stratified instances from a CSV or TXT file.
    """
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_csv(file_path, delim_whitespace=True)
    return df


# -------------------- Extract class --------------------
def extract_class(instance_name):
    """
    Extracts the class information from an instance name.
    The class is represented as (a, X), where a is the prefix, 
    and X is the first number after BPKP (ignoring K, Y, and Z).
    """
    try:
        # Split the instance into the prefix ('a') and the BPPC part
        part_a, rest = instance_name.split('/')
        parts = rest.split('_')
        X = parts[1]  # The first number after 'BPKP'
        return (part_a, X)  # We care about 'a' (prefix) and 'X' (number)
    except:
        return ("unknown", "unknown")


def is_invalid_instance(instance_name):
    """
    Check if the instance matches the invalid pattern, where Y = 0.
    """
    try:
        # Split the instance name into parts
        part_a, rest = instance_name.split('/')
        parts = rest.split('_')
        
        # We expect the pattern to be something like: a/BPKP_X_Y_Z
        X = parts[1]  # The first number after 'BPKP'
        Y = parts[2]  # The second number (Y should not be 0)
        
        # Check if Y = 0 (invalid instance)
        return Y == "0"
    except:
        return False


# -------------------- Stratified sampling --------------------
def stratified_sample(df, n_per_class):
    strata = {}

    # Group instances by class
    for _, row in df.iterrows():
        inst = row["Instance"]
        cls = extract_class(inst)

        if cls not in strata:
            strata[cls] = []
        strata[cls].append(inst)  # store only instance name

    sampled_instances = []

    # Sample from each stratum
    for cls, instances in strata.items():
        # Skip invalid instances and only sample from valid ones
        valid_instances = [inst for inst in instances if not is_invalid_instance(inst)]
        
        if len(valid_instances) <= n_per_class:
            sampled = valid_instances
        else:
            sampled = random.sample(valid_instances, n_per_class)

        sampled_instances.extend(sampled)

        print(f"Class {cls}: selected {len(sampled)} out of {len(valid_instances)}")

    # Return as DataFrame with a single column
    return pd.DataFrame(sampled_instances, columns=["Instance"])


# -------------------- Save outputs --------------------
def save_outputs(df, prefix):
    """
    Save the stratified sample dataframe to CSV and TXT files.
    """
    csv_file = prefix + ".csv"
    txt_file = prefix + ".txt"

    df.to_csv(csv_file, index=False)

    # TXT without header
    df.to_csv(txt_file, index=False, header=False)

    print(f"\nSaved:")
    print(f" - {csv_file}")
    print(f" - {txt_file}")

In [4]:
df = read_instances(INPUT_FILE)

sampled_df = stratified_sample(df, N_PER_CLASS)

save_outputs(sampled_df, OUTPUT_PREFIX)

Class ('u', '1'): selected 5 out of 90
Class ('u', '2'): selected 5 out of 90
Class ('u', '3'): selected 5 out of 90
Class ('u', '4'): selected 5 out of 90
Class ('t', '5'): selected 5 out of 90
Class ('t', '6'): selected 5 out of 90
Class ('t', '7'): selected 5 out of 90
Class ('t', '8'): selected 5 out of 90
Class ('d', '1'): selected 5 out of 90
Class ('d', '2'): selected 5 out of 90
Class ('d', '3'): selected 5 out of 90
Class ('d', '4'): selected 5 out of 90
Class ('da', '1'): selected 5 out of 90
Class ('da', '2'): selected 5 out of 90
Class ('da', '3'): selected 5 out of 90
Class ('da', '4'): selected 5 out of 90
Class ('ta', '5'): selected 5 out of 90
Class ('ta', '6'): selected 5 out of 90
Class ('ta', '7'): selected 5 out of 90
Class ('ta', '8'): selected 5 out of 90
Class ('ua', '1'): selected 5 out of 90
Class ('ua', '2'): selected 5 out of 90
Class ('ua', '3'): selected 5 out of 90
Class ('ua', '4'): selected 5 out of 90

Saved:
 - stratified_sample.csv
 - stratified_sampl